# Exploratory data analysis (EDA) of apartments data - City Apartments

This notebook analyzes apartments in urban/city areas of the canton of Zürich.
City areas are defined as municipalities with population density (pop_dens) at or above the 90% quantile.

## Libraries and settings

In [ ]:
# Libraries
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pylab as py

# seaborn graphics settings
sns.set(color_codes=True)

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Show current working directory
print(os.getcwd())

## Univariate non-graphical exploratory data analysis (EDA)

### Importing the enriched apartment data

In [ ]:
# Read and select variables
df_orig = pd.read_csv("apartments_data_enriched.csv")[['web-scraper-order',
                                                        'address_raw', 'lat', 'lon',
                                                        'bfs_number', 'bfs_name', 'rooms', 
                                                        'area', 'luxurious', 'price', 
                                                        'price_per_m2', 'pop', 'pop_dens',
                                                        'emp', 'frg_pct', 'mean_taxable_income']]
df_orig = df_orig.drop_duplicates()
df_orig = df_orig.dropna()
df_orig.head(5)

### Quantiles original values

In [ ]:
df_orig[['price','rooms', 'area', 'price_per_m2', 'pop_dens']].quantile(q=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]).round(2)

### Filter apartments - CITY AREAS (pop_dens >= 90% quantile = 4778.99)

In [ ]:
# Filter apartments for city areas (pop_dens >= 90% quantile)
df = df_orig.loc[df_orig['pop_dens'] >= 4778.99]
print(f"Number of apartments in city areas: {len(df)}")

### Shape (number of rows and columns)

In [ ]:
print(df.shape)

### Data types

In [ ]:
df.dtypes

### Summary statistics of numeric variables

In [ ]:
df.describe()

### Statistical measures (min, max, std, mean, median, count) for selected variables

In [ ]:
# Price
print('Price:', 'Count:', round(df.price.count(), 1), 'Min:', round(df.price.min(), 1),
      'Max:', round(df.price.max(), 1), 'Mean:', round(df.price.mean(), 1),
      'Median:', round(df.price.median(), 1), 'Std:', round(df.price.std(), 1))
# Area
print('Area:', 'Count:', round(df.area.count(), 1), 'Min:', round(df.area.min(), 1),
      'Max:', round(df.area.max(), 1), 'Mean:', round(df.area.mean(), 1),
      'Median:', round(df.area.median(), 1), 'Std:', round(df.area.std(), 1))

### Skewness

In [ ]:
df[['price','rooms', 'area']].skew()

### Kurtosis

In [ ]:
df[['price','rooms', 'area']].kurtosis()

### Extreme values

In [ ]:
# Low costs apartments
df[df['price_per_m2'] <= 10]

In [ ]:
# Very expensive apartments
df[df['price_per_m2'] >= 100]

### Get a list of categories of categorical variable

In [ ]:
np.array(pd.Categorical(df['bfs_name']).categories)

## Multivariate non-graphical exploratory data analysis (EDA)

### Cross-tabulation

In [ ]:
pd.crosstab(df['luxurious'], df['rooms'])

### Pivot tables

In [ ]:
pd.pivot_table(df[['rooms', 'price', 'price_per_m2', 'area', 'luxurious']],
               index=['rooms', 'luxurious'], values=['price', 'price_per_m2', 'area'],
               aggfunc=(np.mean, 'count'))

### Correlation matrix

In [ ]:
corr = df[['rooms', 'area', 'price', 'price_per_m2', 'pop_dens', 'frg_pct']].cov().corr()
corr

### Covariance matrix

In [ ]:
cov = df[['rooms', 'area', 'price', 'price_per_m2', 'pop_dens', 'frg_pct']].cov()
cov

## Univariate graphical exploratory data analysis (EDA)

### Boxplot (seaborn) - Price per m2

In [ ]:
plt.figure(figsize=(8,1.2))
plt.ticklabel_format(style='plain')
plt.title('Boxplot of price per m2 - City Apartments', fontsize=12)
sns.boxplot(x=df['price_per_m2'], color="lightblue")

### Boxplot (seaborn) - Area

In [ ]:
plt.figure(figsize=(8,1.2))
plt.ticklabel_format(style='plain')
plt.title('Boxplot of area - City Apartments', fontsize=12)
sns.boxplot(x=df['area'], color="lightblue")

### Histogram (matplotlib) - Price per m2

In [ ]:
fig = plt.figure(figsize=(7,4))
n, bins, patches = plt.hist(x=df['price_per_m2'], bins=25, color='#1E90FF', alpha=0.5, rwidth=0.95)
plt.grid(True)
plt.ticklabel_format(style='plain')
plt.xlabel('price_per_m2', fontsize=10, labelpad=10)
plt.ylabel('Number of apartments', fontsize=10, labelpad=10)
plt.title('Histogram of price_per_m2 - City Apartments', fontsize=12, pad=10)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.show()

### Histogram (matplotlib) - Area

In [ ]:
fig = plt.figure(figsize=(7,4))
n, bins, patches = plt.hist(x=df['area'], bins=25, color='#1E90FF', alpha=0.5, rwidth=0.95)
plt.grid(True)
plt.ticklabel_format(style='plain')
plt.xlabel('area (m2)', fontsize=10, labelpad=10)
plt.ylabel('Number of apartments', fontsize=10, labelpad=10)
plt.title('Histogram of area - City Apartments', fontsize=12, pad=10)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.show()

### Density plot (seaborn)

In [ ]:
plt.figure(figsize=(7,4))
sns.distplot(df['price_per_m2'], hist=True, kde=True, bins=20, color='blue',
             hist_kws={'edgecolor':'black'}, kde_kws={'linewidth': 2})
plt.title('Density plot price per m2 - City Apartments', fontsize=12, pad=10)
plt.xlabel('price_per_m2', fontsize=12, labelpad=10)
plt.ylabel('Density', fontsize=12, labelpad=10)
plt.grid(True)
plt.show()

### Quantile-Quantile (QQ-) plot

In [ ]:
x = 'area'
df_qq = df.copy()
df_qq['var'] = (df[x]-df[x].mean()) / df[x].std()
print(df_qq.sort_values('var')[['area', 'var']])
sm.qqplot(df_qq['var'], line='45')
py.show()

### Barchart (matplotlib)

In [ ]:
df_bar = df['rooms'].value_counts().nlargest(15).sort_values(ascending=True)
napart = list(df_bar.values)
index = list(df_bar.index.values)
y_pos = np.arange(len(index))
fig, ax = plt.subplots(figsize=(7,4))
ax.barh(y_pos, napart, align='center', color='b', alpha=0.8)
ax.set_yticks(y_pos, index)
ax.set_xlabel('Number of apartments', fontsize=10)
ax.set_ylabel('Rooms', fontsize=10)
ax.set_title('Number of apartments by rooms - City Apartments', fontsize=12)
plt.show()

## Multivariate graphical exploratory data analysis (EDA)

### Scatterplot (matplotlib)

In [ ]:
plt.figure(figsize=(7,4))
plt.scatter(df['area'], df['price'], color="blue", alpha=1.0, s=10)
plt.title('Scatterplot - City Apartments', fontsize=12)
plt.xlabel('area (m2)')
plt.ylabel('price (CHF)')
plt.show()

### Scatterplot (matplotlib) with regression line

In [ ]:
df_sub = df.loc[(df.price >= 1000)]
print(df_sub.shape)
plt.figure(figsize=(7,4))
plt.plot(df_sub.area, df_sub.price, 'o', markersize=3.5, color="blue")
b, a = np.polyfit(df_sub.area, df_sub.price, 1)
print(f"Slope: {b}")
print(f"Intercept: {a}")
plt.plot(df_sub.area, b*df_sub.area + a, linewidth=1, linestyle='dashed', color='darkred')
plt.title('Scatterplot with regression line - City Apartments', fontsize=12)
plt.ylabel('price', fontsize=12)
plt.xlabel('area', fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.show()

### Scatterplot-matrix (seaborn)

In [ ]:
sns.set(style="ticks", font_scale=0.8)
g = sns.PairGrid(df[['rooms', 'area', 'price', 'price_per_m2', 'pop_dens', 'frg_pct']], height=1.2, aspect=1)
g.map_upper(sns.scatterplot, color='darkblue', s=10)
g.map_lower(sns.scatterplot, color='darkblue', s=10)
g.map_diag(plt.hist, color='brown')

### Correlation heatmap (seaborn)

In [ ]:
sns.set(font_scale=0.8)
plt.figure(figsize=(7,4))
corr = df[['rooms', 'area', 'price', 'price_per_m2', 'pop_dens', 'frg_pct']].corr().round(2)
sns.heatmap(corr, cmap="BrBG", annot=True)

### Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')